# POST SSE 이벤트 설계

POST 본문으로 질문을 보내고 네이티브 SSE의 `event`, `id`, `data` 필드를 확인합니다.

## 실행 환경 준비

현재 폴더나 상위 폴더에서 `labs/part7/ch30-fastapi-serving`을 찾아 작업 폴더로 사용합니다. 비밀 키를 출력하지 않습니다.

In [1]:
import os
from pathlib import Path
start = Path.cwd().resolve()
lab = next((p for p in (start, *start.parents)
    if (p / 'agent_service').is_dir()), None)
assert lab is not None, 'agent_service가 있는 Ch30 실습 폴더에서 실행하세요.'
os.chdir(lab)
print('실습 폴더:', Path.cwd().name)

실습 폴더: ch30-fastapi-serving


## 기본 실행

TestClient가 실제 ASGI 스트림을 끝까지 읽어 전송 형식을 보여 줍니다.

In [2]:
from fastapi.testclient import TestClient
from agent_service.app import app
client = TestClient(app)
headers = {'Authorization': 'Bearer local-demo-token'}
response = client.post('/v1/chat/stream', headers=headers,
    json={'question': 'SSE 확인', 'thread_id': 'sse-1'})
print(response.status_code, response.headers['content-type'])
assert response.status_code == 200

200 text/event-stream; charset=utf-8


## 이벤트 필드 관찰

토큰 이벤트는 재연결 위치에 쓸 `id`를 갖고 완료 이벤트는 별도 타입입니다.

In [3]:
import re
blocks = [b for b in re.split(r'\r?\n\r?\n', response.text) if b]
print(blocks[0])
print(blocks[-1])
assert blocks[0].splitlines()[0] == 'event: token'
assert 'id: sse-1:0' in blocks[0]
assert blocks[-1].splitlines()[0] == 'event: done'

event: token
data: {"token": "\uc9c8\ubb38\uc744 "}
id: sse-1:0
event: done
data: {"thread_id": "sse-1"}


## 반복 실험과 결과 해석

문장 길이만 바꾸면 토큰 이벤트 수는 달라지지만 완료 이벤트는 한 번입니다.

In [4]:
for text in ['한', '두 단어']:
    res = client.post('/v1/chat/stream', headers=headers,
        json={'question': text, 'thread_id': 'count'})
    token_count = res.text.count('event: token')
    done_count = res.text.count('event: done')
    print(text, token_count, done_count)
    assert done_count == 1

한 3 1
두 단어 4 1
